In [0]:
from pyspark.sql import functions as F

doctors_bronze = spark.table(
    "healthcare.default.bronze_doctors"
)

print("Doctors:", doctors_bronze.count())

Doctors: 10


In [0]:
display(doctors_bronze)

doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,2fd108434c9a1abca8ee2737545b65c5821dbbecf358ad2f1512e6c897fef35d,BRONZE
D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,5a13a9beb52ea33d58dd271aab12aa727346b2991cc8d2bc6e914edb26526d92,BRONZE
D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,a57399a935744dcdb43182b557c416c61f69873ca546b319b2ae58a376f4756f,BRONZE
D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,537e428029b87c3e1ddc93b1da665d8859b77aed5a6836c238abab4954054211,BRONZE
D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,382a00030c2f49febb0f65dba0c7b8d75732e4ffcd194dd4eef46158578f1918,BRONZE
D006,Alex,Davis,Pediatrics,6570137231,23,Central Hospital,dr.alex.davis@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,5a4ee9fb29deb16f3c2cb56b241c10ba8a4b1582209f16a58b919e04121fcc61,BRONZE
D007,Robert,Davis,Oncology,8217493115,26,Westside Clinic,dr.robert.davis@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,a3f6bc87dd0e29cc01bdd5978b2809a7c34ecd3770df10012475d7e50771b2cf,BRONZE
D008,Linda,Brown,Dermatology,9069162601,5,Westside Clinic,dr.linda.brown@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,8f33e2856b4bd096bd8fa18289d7717b97af798bd97473e31fe4e38480e1e755,BRONZE
D009,Sarah,Smith,Pediatrics,7387087517,26,Central Hospital,dr.sarah.smith@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,22eeaeffa938c89c7e5c7387605f929a52088af39d49979a9da048556717268c,BRONZE
D010,Linda,Wilson,Oncology,6176383634,21,Eastside Clinic,dr.linda.wilson@hospital.com,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,dcd6af3ca88c8a8fe020c530616c3c27724f056593c5b3f06056eb03d9994a91,BRONZE


In [0]:
duplicate_doctors = (
    doctors_bronze
    .groupBy("doctor_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate doctor IDs:",
    duplicate_doctors.count()
)

display(duplicate_doctors)

Duplicate doctor IDs: 0


doctor_id,count


In [0]:
doctor_nulls = doctors_bronze.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in doctors_bronze.columns
])

display(doctor_nulls)

doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
display(
    doctors_bronze
    .groupBy("specialization")
    .count()
    .orderBy("specialization")
)

specialization,count
Dermatology,3
Oncology,2
Pediatrics,5


In [0]:
display(
    doctors_bronze
    .groupBy("hospital_branch")
    .count()
    .orderBy("hospital_branch")
)

hospital_branch,count
Central Hospital,4
Eastside Clinic,3
Westside Clinic,3


In [0]:
display(
    doctors_bronze.select(
        F.min("years_experience").alias("minimum_experience"),
        F.max("years_experience").alias("maximum_experience"),
        F.avg("years_experience").alias("average_experience")
    )
)

minimum_experience,maximum_experience,average_experience
5,28,21.5


In [0]:
# Clean and validate doctor data

doctors_checked = (
    doctors_bronze

    # Clean text fields
    .withColumn(
        "first_name_clean",
        F.trim(F.col("first_name"))
    )

    .withColumn(
        "last_name_clean",
        F.trim(F.col("last_name"))
    )

    .withColumn(
        "specialization_clean",
        F.initcap(F.trim(F.col("specialization")))
    )

    .withColumn(
        "hospital_branch_clean",
        F.initcap(F.trim(F.col("hospital_branch")))
    )

    .withColumn(
        "email_clean",
        F.lower(F.trim(F.col("email")))
    )

    # Convert phone number to string
    .withColumn(
        "phone_number_clean",
        F.col("phone_number").cast("string")
    )

    # Validation rules
    .withColumn(
        "valid_doctor_id",
        F.col("doctor_id").isNotNull()
    )

    .withColumn(
        "valid_names",
        (
            F.col("first_name_clean").isNotNull()
            &
            F.col("last_name_clean").isNotNull()
        )
    )

    .withColumn(
        "valid_specialization",
        F.col("specialization_clean").isNotNull()
    )

    .withColumn(
        "valid_experience",
        (
            F.col("years_experience").isNotNull()
            &
            (F.col("years_experience") >= 0)
        )
    )

    .withColumn(
        "valid_email",
        F.col("email_clean").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        )
    )
)

display(
    doctors_checked.select(
        "doctor_id",
        "first_name_clean",
        "last_name_clean",
        "specialization_clean",
        "years_experience",
        "hospital_branch_clean",
        "email_clean",
        "valid_doctor_id",
        "valid_names",
        "valid_specialization",
        "valid_experience",
        "valid_email"
    )
)

doctor_id,first_name_clean,last_name_clean,specialization_clean,years_experience,hospital_branch_clean,email_clean,valid_doctor_id,valid_names,valid_specialization,valid_experience,valid_email
D001,David,Taylor,Dermatology,17,Westside Clinic,dr.david.taylor@hospital.com,true,true,true,true,true
D002,Jane,Davis,Pediatrics,24,Eastside Clinic,dr.jane.davis@hospital.com,true,true,true,true,true
D003,Jane,Smith,Pediatrics,19,Eastside Clinic,dr.jane.smith@hospital.com,true,true,true,true,true
D004,David,Jones,Pediatrics,28,Central Hospital,dr.david.jones@hospital.com,true,true,true,true,true
D005,Sarah,Taylor,Dermatology,26,Central Hospital,dr.sarah.taylor@hospital.com,true,true,true,true,true
D006,Alex,Davis,Pediatrics,23,Central Hospital,dr.alex.davis@hospital.com,true,true,true,true,true
D007,Robert,Davis,Oncology,26,Westside Clinic,dr.robert.davis@hospital.com,true,true,true,true,true
D008,Linda,Brown,Dermatology,5,Westside Clinic,dr.linda.brown@hospital.com,true,true,true,true,true
D009,Sarah,Smith,Pediatrics,26,Central Hospital,dr.sarah.smith@hospital.com,true,true,true,true,true
D010,Linda,Wilson,Oncology,21,Eastside Clinic,dr.linda.wilson@hospital.com,true,true,true,true,true


In [0]:
doctor_validation_summary = doctors_checked.select(
    F.count("*").alias("total_records"),

    F.sum(
        F.when(~F.col("valid_doctor_id"), 1)
         .otherwise(0)
    ).alias("invalid_doctor_id"),

    F.sum(
        F.when(~F.col("valid_names"), 1)
         .otherwise(0)
    ).alias("invalid_names"),

    F.sum(
        F.when(~F.col("valid_specialization"), 1)
         .otherwise(0)
    ).alias("invalid_specialization"),

    F.sum(
        F.when(~F.col("valid_experience"), 1)
         .otherwise(0)
    ).alias("invalid_experience"),

    F.sum(
        F.when(~F.col("valid_email"), 1)
         .otherwise(0)
    ).alias("invalid_email")
)

display(doctor_validation_summary)

total_records,invalid_doctor_id,invalid_names,invalid_specialization,invalid_experience,invalid_email
10,0,0,0,0,0


In [0]:
silver_doctors = (
    doctors_checked

    .filter(
        F.col("valid_doctor_id")
        & F.col("valid_names")
        & F.col("valid_specialization")
        & F.col("valid_experience")
        & F.col("valid_email")
    )

    .select(
        "doctor_id",

        F.col("first_name_clean").alias("first_name"),

        F.col("last_name_clean").alias("last_name"),

        F.col("specialization_clean").alias("specialization"),

        F.col("phone_number_clean").alias("phone_number"),

        "years_experience",

        F.col("hospital_branch_clean").alias("hospital_branch"),

        F.col("email_clean").alias("email"),

        # Validation metadata
        "valid_doctor_id",
        "valid_names",
        "valid_specialization",
        "valid_experience",
        "valid_email",

        # Pipeline lineage
        "_batch_id",
        "_source_id",
        "_source_name",
        "_source_file_name",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_record_hash"
    )
)

print(
    "Silver doctor records:",
    silver_doctors.count()
)

display(silver_doctors)

Silver doctor records: 10


doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email,valid_doctor_id,valid_names,valid_specialization,valid_experience,valid_email,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash
D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,2fd108434c9a1abca8ee2737545b65c5821dbbecf358ad2f1512e6c897fef35d
D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,5a13a9beb52ea33d58dd271aab12aa727346b2991cc8d2bc6e914edb26526d92
D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,a57399a935744dcdb43182b557c416c61f69873ca546b319b2ae58a376f4756f
D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,537e428029b87c3e1ddc93b1da665d8859b77aed5a6836c238abab4954054211
D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,382a00030c2f49febb0f65dba0c7b8d75732e4ffcd194dd4eef46158578f1918
D006,Alex,Davis,Pediatrics,6570137231,23,Central Hospital,dr.alex.davis@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,5a4ee9fb29deb16f3c2cb56b241c10ba8a4b1582209f16a58b919e04121fcc61
D007,Robert,Davis,Oncology,8217493115,26,Westside Clinic,dr.robert.davis@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,a3f6bc87dd0e29cc01bdd5978b2809a7c34ecd3770df10012475d7e50771b2cf
D008,Linda,Brown,Dermatology,9069162601,5,Westside Clinic,dr.linda.brown@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,8f33e2856b4bd096bd8fa18289d7717b97af798bd97473e31fe4e38480e1e755
D009,Sarah,Smith,Pediatrics,7387087517,26,Central Hospital,dr.sarah.smith@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,22eeaeffa938c89c7e5c7387605f929a52088af39d49979a9da048556717268c
D010,Linda,Wilson,Oncology,6176383634,21,Eastside Clinic,dr.linda.wilson@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,dcd6af3ca88c8a8fe020c530616c3c27724f056593c5b3f06056eb03d9994a91


In [0]:
silver_doctors.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "healthcare.default.silver_doctors"
    )

print("silver_doctors created successfully.")

silver_doctors created successfully.


In [0]:
display(
    spark.table(
        "healthcare.default.silver_doctors"
    )
)

doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email,valid_doctor_id,valid_names,valid_specialization,valid_experience,valid_email,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash
D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,2fd108434c9a1abca8ee2737545b65c5821dbbecf358ad2f1512e6c897fef35d
D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,5a13a9beb52ea33d58dd271aab12aa727346b2991cc8d2bc6e914edb26526d92
D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,a57399a935744dcdb43182b557c416c61f69873ca546b319b2ae58a376f4756f
D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,537e428029b87c3e1ddc93b1da665d8859b77aed5a6836c238abab4954054211
D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,382a00030c2f49febb0f65dba0c7b8d75732e4ffcd194dd4eef46158578f1918
D006,Alex,Davis,Pediatrics,6570137231,23,Central Hospital,dr.alex.davis@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,5a4ee9fb29deb16f3c2cb56b241c10ba8a4b1582209f16a58b919e04121fcc61
D007,Robert,Davis,Oncology,8217493115,26,Westside Clinic,dr.robert.davis@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,a3f6bc87dd0e29cc01bdd5978b2809a7c34ecd3770df10012475d7e50771b2cf
D008,Linda,Brown,Dermatology,9069162601,5,Westside Clinic,dr.linda.brown@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,8f33e2856b4bd096bd8fa18289d7717b97af798bd97473e31fe4e38480e1e755
D009,Sarah,Smith,Pediatrics,7387087517,26,Central Hospital,dr.sarah.smith@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,22eeaeffa938c89c7e5c7387605f929a52088af39d49979a9da048556717268c
D010,Linda,Wilson,Oncology,6176383634,21,Eastside Clinic,dr.linda.wilson@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,dcd6af3ca88c8a8fe020c530616c3c27724f056593c5b3f06056eb03d9994a91


In [0]:
display(
    spark.table(
        "healthcare.default.silver_doctors"
    )
    .groupBy("specialization")
    .count()
    .orderBy("specialization")
)

specialization,count
Dermatology,3
Oncology,2
Pediatrics,5
